# TF-IDF karakter + cosine
**text › retrieval** · notebook `tfidf_char_cosine`

**Kapan dipakai:** Tahan typo/variasi ejaan dan bahasa kaya imbuhan.

**Jebakan umum:** Indeks lebih besar dan lebih lambat.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/text/classification_en/news4_train.csv"
TEST_PATH  = ROOT / "data_latihan/text/classification_en/news4_test.csv"
TEXT_COL   = "text"
LABEL_COL  = "label"            # hanya untuk EVALUASI; algoritma tidak melihatnya
LANG       = "en"
USE_STOPWORDS = True
SEED = 42
MODEL_KEY  = "tfidf_char_cosine"
OUT = out_dir("text", "retrieval", MODEL_KEY)
set_seed(SEED)
from superdb.text import ID_STOPWORDS
STOP = (ID_STOPWORDS if LANG == "id" else "english") if USE_STOPWORDS else None
N_QUERIES, TOPK = 150, 10

## 2. Korpus dan query
**Korpus** = dokumen latih. **Query** = dokumen uji. Dokumen **relevan** untuk sebuah query = dokumen korpus berkategori sama (proksi relevansi yang jelas dan bisa diukur).

In [ ]:
from superdb.text import clean_text
corpus_df = pd.read_csv(TRAIN_PATH)
q_df = pd.read_csv(TEST_PATH).merge(pd.read_csv(str(TEST_PATH).replace("_test.csv", "_test_labels.csv")), on="id").sample(N_QUERIES, random_state=SEED)
D = corpus_df[TEXT_COL].map(clean_text).tolist(); Q = q_df[TEXT_COL].map(clean_text).tolist()
d_lab, q_lab = corpus_df[LABEL_COL].to_numpy(), q_df[LABEL_COL].to_numpy()
print(len(D), "dokumen korpus;", len(Q), "query")

## 3. Skor kemiripan query × dokumen

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
scorer = "tfidf"
from sklearn.metrics.pairwise import cosine_similarity
if scorer == "bm25":
    import re
    tok = lambda s: re.findall(r"\w+", s.lower())
    bm = BM25Okapi([tok(d) for d in D])
    scores = np.vstack([bm.get_scores(tok(q)) for q in Q])
else:
    Dm = vec.fit_transform(D); Qm = vec.transform(Q)
    scores = cosine_similarity(Qm, Dm)
scores.shape

## 4. Metrik ranking
**P@10**: dari 10 teratas, berapa yang relevan. **MAP@10** dan **nDCG@10**: memperhitungkan urutan. **MRR**: posisi hasil relevan pertama. Garis dasar acak = proporsi kategori.

In [ ]:
from superdb.text_extra import retrieval_metrics
metrics = retrieval_metrics(scores, q_lab, d_lab, k=TOPK)
rand = float(np.mean([np.mean(d_lab == l) for l in q_lab]))
display(show_table({**metrics, "acak P@k (dasar)": rand}))

## 5. Contoh query

In [ ]:
i = 0; top = np.argsort(-scores[i])[:3]
print("QUERY (", q_lab[i], "):", Q[i][:160], "...\n")
for r, j in enumerate(top, 1):
    print(f"#{r} skor={scores[i, j]:.3f} kategori={d_lab[j]}:", D[j][:140], "...")
save_json({"metrics": metrics, "random_baseline": rand}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan BM25, TF-IDF, LSA, dan embedding (heavy) pada **query yang sama**.
- Hasil terbaik biasanya hibrida: skor leksikal (BM25) digabung skor embedding.
- Data lomba: relevansi nyata datang dari file qrels, ganti `retrieval_metrics` dengan label itu.